# Encoding categorical variables

- **Numerical variable :**
  - quantity represented by real or integer numbers.
  - can be naturally handled by machine learning algorithms that are typically composed of a sequence of arithmetic instructions 
- **Categorical variable :**
  - Have discrete values
  - mostly represented by strings 

In [25]:
import pandas as pd
adult_census = pd.read_csv('data/adult_data.csv')
adult_census = adult_census.drop(columns = "education-num")

target_name = "class"
target = adult_census[target_name]
data = adult_census.drop(columns =[target_name])

In [26]:
data.head()

,age,workclass,fnlwgt,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25.0,b'Private',226802.0,b'11th',b'Never-married',b'Machine-op-inspct',b'Own-child',b'Black',b'Male',0.0,0.0,40.0,b'United-States'
1,38.0,b'Private',89814.0,b'HS-grad',b'Married-civ-spouse',b'Farming-fishing',b'Husband',b'White',b'Male',0.0,0.0,50.0,b'United-States'
2,28.0,b'Local-gov',336951.0,b'Assoc-acdm',b'Married-civ-spouse',b'Protective-serv',b'Husband',b'White',b'Male',0.0,0.0,40.0,b'United-States'
3,44.0,b'Private',160323.0,b'Some-college',b'Married-civ-spouse',b'Machine-op-inspct',b'Husband',b'Black',b'Male',7688.0,0.0,40.0,b'United-States'
4,18.0,b'?',103497.0,b'Some-college',b'Never-married',b'?',b'Own-child',b'White',b'Female',0.0,0.0,30.0,b'United-States'


- `native-country` is the categorial variable here

In [27]:
data["native-country"].value_counts().sort_index()

native-country
b'?'                               857
b'Cambodia'                         28
b'Canada'                          182
b'China'                           122
b'Columbia'                         85
b'Cuba'                            138
b'Dominican-Republic'              103
b'Ecuador'                          45
b'El-Salvador'                     155
b'England'                         127
b'France'                           38
b'Germany'                         206
b'Greece'                           49
b'Guatemala'                        88
b'Haiti'                            75
b'Holand-Netherlands'                1
b'Honduras'                         20
b'Hong'                             30
b'Hungary'                          19
b'India'                           151
b'Iran'                             59
b'Ireland'                          37
b'Italy'                           105
b'Jamaica'                         106
b'Japan'                            92
b'Laos'   

In [28]:
data.dtypes

age               float64
workclass          object
fnlwgt            float64
education          object
marital-status     object
occupation         object
relationship       object
race               object
sex                object
capital-gain      float64
capital-loss      float64
hours-per-week    float64
native-country     object
dtype: object

In [29]:
from sklearn.compose import make_column_selector as selector

categorical_columns_selector = selector(dtype_include=object)
categorical_columns = categorical_columns_selector(data)
data_categorical = data[categorical_columns]
data_categorical.head()

,workclass,education,marital-status,occupation,relationship,race,sex,native-country
0,b'Private',b'11th',b'Never-married',b'Machine-op-inspct',b'Own-child',b'Black',b'Male',b'United-States'
1,b'Private',b'HS-grad',b'Married-civ-spouse',b'Farming-fishing',b'Husband',b'White',b'Male',b'United-States'
2,b'Local-gov',b'Assoc-acdm',b'Married-civ-spouse',b'Protective-serv',b'Husband',b'White',b'Male',b'United-States'
3,b'Private',b'Some-college',b'Married-civ-spouse',b'Machine-op-inspct',b'Husband',b'Black',b'Male',b'United-States'
4,b'?',b'Some-college',b'Never-married',b'?',b'Own-child',b'White',b'Female',b'United-States'


In [30]:
data_categorical.shape

(48842, 8)

## Ordinal encoding

In [31]:
from sklearn.preprocessing import OrdinalEncoder
education_column = data_categorical[["education"]]
encoder = OrdinalEncoder().set_output(transform = "pandas")
education_encoded = encoder.fit_transform(education_column)
education_encoded.head()

,education
0,1.0
1,11.0
2,7.0
3,15.0
4,15.0


In [32]:
encoder.categories_

[array(["b'10th'", "b'11th'", "b'12th'", "b'1st-4th'", "b'5th-6th'",
        "b'7th-8th'", "b'9th'", "b'Assoc-acdm'", "b'Assoc-voc'",
        "b'Bachelors'", "b'Doctorate'", "b'HS-grad'", "b'Masters'",
        "b'Preschool'", "b'Prof-school'", "b'Some-college'"], dtype=object)]

- each category is encoded with a different no.
- `encoder.categories_` gives the order in which the categories were encoded
- > 10th will be given value 0 <br>
  > 11th will be given value 1

In [33]:
data_encoded = encoder.fit_transform(data_categorical)

In [34]:
data_encoded.head()

,workclass,education,marital-status,occupation,relationship,race,sex,native-country
0,4.0,1.0,4.0,7.0,3.0,2.0,1.0,39.0
1,4.0,11.0,2.0,5.0,0.0,4.0,1.0,39.0
2,2.0,7.0,2.0,11.0,0.0,4.0,1.0,39.0
3,4.0,15.0,2.0,7.0,0.0,2.0,1.0,39.0
4,0.0,15.0,4.0,0.0,3.0,4.0,0.0,39.0


In [35]:
data_encoded.shape

(48842, 8)

- Encoding is done in lexicographical order.
- To do custom order
- By default it would have been *High->0, Med->1, Low->2*
- But after custom order it will be *Low->0, Med->1, High->2*
-  ``` python
    from sklearn.preprocessing import OrdinalEncoder
    encoder = OrdinalEncoder(categories=[["Low", "Medium", "High"]])  # Custom order
    education_column = [["Low"], ["Medium"], ["High"]]
    education_encoded = encoder.fit_transform(education_column)
    print(education_encoded)

  ```

- If a categorical variable does not carry any meaningful order information then this encoding might be misleading to downstream statistical models and you might consider using one-hot encoding instead.

## OneHotEncoder

- For a given feature, it creates as many new columns as there are possible categories. For a given sample, the value of the column corresponding to the category is set to 1 while all the columns of the other categories are set to 0

In [36]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False).set_output(transform = "pandas")
education_encoded = encoder.fit_transform(education_column)
education_encoded.head()

,education_b'10th',education_b'11th',education_b'12th',education_b'1st-4th',education_b'5th-6th',education_b'7th-8th',education_b'9th',education_b'Assoc-acdm',education_b'Assoc-voc',education_b'Bachelors',education_b'Doctorate',education_b'HS-grad',education_b'Masters',education_b'Preschool',education_b'Prof-school',education_b'Some-college'
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [37]:
education_encoded.shape

(48842, 16)

In [38]:
data_categorical.head()

,workclass,education,marital-status,occupation,relationship,race,sex,native-country
0,b'Private',b'11th',b'Never-married',b'Machine-op-inspct',b'Own-child',b'Black',b'Male',b'United-States'
1,b'Private',b'HS-grad',b'Married-civ-spouse',b'Farming-fishing',b'Husband',b'White',b'Male',b'United-States'
2,b'Local-gov',b'Assoc-acdm',b'Married-civ-spouse',b'Protective-serv',b'Husband',b'White',b'Male',b'United-States'
3,b'Private',b'Some-college',b'Married-civ-spouse',b'Machine-op-inspct',b'Husband',b'Black',b'Male',b'United-States'
4,b'?',b'Some-college',b'Never-married',b'?',b'Own-child',b'White',b'Female',b'United-States'


In [39]:
data_encoded = encoder.fit_transform(data_categorical)
data_encoded.head()

,workclass_b'?',workclass_b'Federal-gov',workclass_b'Local-gov',workclass_b'Never-worked',workclass_b'Private',workclass_b'Self-emp-inc',workclass_b'Self-emp-not-inc',workclass_b'State-gov',workclass_b'Without-pay',education_b'10th',...,native-country_b'Portugal',native-country_b'Puerto-Rico',native-country_b'Scotland',native-country_b'South',native-country_b'Taiwan',native-country_b'Thailand',native-country_b'Trinadad&Tobago',native-country_b'United-States',native-country_b'Vietnam',native-country_b'Yugoslavia'
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [40]:
data_encoded.shape

(48842, 102)

- In general OneHotEncoder is the encoding strategy used when the downstream models are linear models while OrdinalEncoder is often a good strategy with tree-based models

- Holand-Netherlands in native-country has only 1 occurence
-  This will be a problem during cross-validation: if the sample ends up in the test set during splitting then the classifier would not have seen the category during training and would not be able to encode it
-  set the parameter `handle_unknown="ignore"`, i.e. if an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros;

In [41]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    OneHotEncoder(handle_unknown = "ignore"), LogisticRegression(max_iter = 500)
)

In [42]:
from sklearn.model_selection import cross_validate
cv_results = cross_validate(model, data_categorical, target)
cv_results

{'fit_time': array([0.41940904, 0.37998033, 0.4174962 , 0.41235757, 0.38509321]),
 'score_time': array([0.05682898, 0.05999947, 0.05664778, 0.05682707, 0.0560019 ]),
 'test_score': array([0.83232675, 0.83570478, 0.82831695, 0.83292383, 0.83497133])}

In [43]:
scores = cv_results["test_score"]
print(f"The accuracy is: {scores.mean():.3f} ± {scores.std():.3f}")

The accuracy is: 0.833 ± 0.003


- Using only numerical data, accuracy was `0.800 ± 0.003`

### Using ordinal encoder

In [23]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
model = make_pipeline(
    OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value = -1),
    LogisticRegression(max_iter=500)
)

In [24]:
from sklearn.model_selection import cross_validate

cv_results = cross_validate(model, data_categorical, target)

scores = cv_results["test_score"]
print(
    "The mean cross-validation accuracy is: "
    f"{scores.mean():.3f} ± {scores.std():.3f}"
)

The mean cross-validation accuracy is: 0.755 ± 0.002


- Thus, OneHotEncoding gave better accuracy
- linear model and OrdinalEncoder are used together only for ordinal categorical features, i.e. features that have a specific ordering. Otherwise, your model would perform poorly.